In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import nltk
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import joblib
import json
import joblib
from kafka import KafkaConsumer, KafkaProducer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

Créatiioion d'un modèle de prédiction

In [2]:
# df_idmb = pd.read_csv('imdb.csv', encoding='ISO-8859-1')
# df_idmb = df_idmb[(df_idmb['label'] == 'neg') | (df_idmb['label'] == 'pos')]
# # df_idmb

# mot_vide = set(stopwords.words('english'))

# lemmatizer = WordNetLemmatizer()


# def nettoyage(data):
#     data = str(data).lower()
    
#     data = re.sub(r"[-–—’'\"“”]", " ",data)
#     data = re.sub(r"[^a-z\s]", " ",data)
#     words = data.split()
#     text_lematise = [lemmatizer.lemmatize(word) for word in words if word not in mot_vide and len(word) > 2]

#     return " ".join(text_lematise)

# df_idmb['review'] = df_idmb['review'].apply(nettoyage)

# df = df_idmb[['review', 'label']]

# x = df['review']
# y = df['label']

# encoder = LabelEncoder()
# y = encoder.fit_transform(y)
# # print(y)

# X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2)



# pipeline = Pipeline([
#     ('Tf-idf', TfidfVectorizer()),  
#     ('logreg', LogisticRegression()) 
# ])

# pipeline.fit(X_train, y_train)
import pandas as pd
import sklearn
import re
from wordcloud import WordCloud
import numpy 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pickle
import nltk
import joblib
# nltk.download()
from  nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, accuracy_score, recall_score, classification_report

# nltk.download('stopwords')
# nltk.download('punkt')
# nltk.download('wordnet')

import nltk
print(nltk.data.path)

lemmatizer=WordNetLemmatizer()
content =pd.read_csv('imdb.csv', encoding='ISO-8859-1')


# print(content)
# data=content.lower()
testing_data=content[["type","review","label"]]
testing_data=testing_data[testing_data["type"]=="test"]
testing_data=testing_data[testing_data["label"]!='unsup']

# (testing_data)
train_data=content[["type","review","label"]]
train_data=train_data[train_data["type"]=="train"]
train_data=train_data[train_data["label"]!='unsup']

def cleaner(arg): 
    # train_data["review"]=train_data["review"].str.lower()
    arg=arg.lower()
    arg=re.sub(r"[^a-z]", " ", arg)
    arg=re.sub(r'<[^>]+>', " ", arg)
    # data=re.sub(r"[^a-z]", " ", train_data["review"])

    # print(train_data)
    # print(data)
    stop_words= set(stopwords.words('english'))
    tokens=word_tokenize(arg)
    # print(tokens)
    filtered_tokens=[word for word in tokens if word not in stop_words and len(word)>2 ]
    # print(filtered_tokens)
    lemmatized_word=[lemmatizer.lemmatize(word) for word in filtered_tokens]

    joined_lemmatized_word=" ".join(lemmatized_word)
    return joined_lemmatized_word

x_train=train_data["review"]
x_test=testing_data["review"]

x_train= x_train.apply(cleaner)
x_test=x_test.apply(cleaner)

model = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', KNeighborsClassifier())
])
model.fit(x_train, train_data['label'])
y_pred =model.predict(x_test)
y_pred


y_test=testing_data['label']
precision =accuracy_score(y_pred, y_test)
print(precision)

joblib.dump(model, "analyse_sentiment.pkl")

['C:\\Users\\PC/nltk_data', 'c:\\Users\\PC\\OneDrive - Epitech\\rendu\\C-DAT-300-ABJ-2-1-endtoendml-7\\.venv\\nltk_data', 'c:\\Users\\PC\\OneDrive - Epitech\\rendu\\C-DAT-300-ABJ-2-1-endtoendml-7\\.venv\\share\\nltk_data', 'c:\\Users\\PC\\OneDrive - Epitech\\rendu\\C-DAT-300-ABJ-2-1-endtoendml-7\\.venv\\lib\\nltk_data', 'C:\\Users\\PC\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data']
0.65052


['analyse_sentiment.pkl']

Création d'un consommateur kafka pour entainer le modèle

In [ ]:


# BROKER = "localhost:9092"       
# TOPIC_ENTREE = "comments"         
# TOPIC_SORTIE = "comments_sentiment_netflix" 
# MODELE_PATH = "analyse_sentiment.pkl"  
# GROUPE_ID = "sentiment-consumer"

# def charger_modele(path):

#     modele = joblib.load(path)
#     return modele

# mot_vide = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()
# def nettoyage(data):
#     data = str(data).lower()
#     data = re.sub(r"[-–—’'\"“”]", " ",data)
#     data = re.sub(r"[^a-z\s]", " ",data)
#     words = data.split()
#     text_lematise = [lemmatizer.lemmatize(word) for word in words if word not in mot_vide and len(word) > 2]

#     return " ".join(text_lematise)

# def analyser_texte(modele, texte):

#     prediction = float(modele.predict([texte])[0])
#     if prediction ==  0 :
#         sentiment = "negatif"
#     else:
#         sentiment = "positif"

#     return {"sentiment": sentiment, "prediction": prediction}



# def consommer_tweets():
#     modele = charger_modele(MODELE_PATH)

#     consommateur = KafkaConsumer(
#         TOPIC_ENTREE,
#         bootstrap_servers=[BROKER],
#         auto_offset_reset="earliest",
#         group_id=GROUPE_ID,
#         value_deserializer=lambda v: v
#     )

#     producteur = KafkaProducer(
#         bootstrap_servers=[BROKER],
#         value_serializer=lambda v: json.dumps(v).encode("utf-8")
#     )

#     try:
#         for msg in consommateur:
#             try:
#                 data = json.loads(msg.value)
#             except Exception:
#                 data = {"text": msg.value}

#             texte_title = data.get("title")
#             texte = data.get("comments")
#             if not texte:
#                 print("Aucun texte trouvé.")
#                 continue
#             title = texte_title
#             textes = nettoyage(texte)

#             resultat = analyser_texte(modele, textes)

#             sortie = {
#                 "title": title,
#                 "comments": textes,
#                 "sentiment": resultat["sentiment"],
#             }

#             print(json.dumps(sortie, ensure_ascii=False))

#             producteur.send(TOPIC_SORTIE, sortie)

#     except KeyboardInterrupt:
#         print("Arrêt du consommateur")
#     finally:
#         consommateur.close()
#         producteur.close()

# if __name__ == "__main__":
#     consommer_tweets()